# AeroAgent: System Deep Dive
### Autonomous AOG Quoting Engine for Ahmed Jet Support

This notebook demonstrates the underlying logic of the **AeroAgent** system. 
AOG (Aircraft on Ground) situations cost airlines thousands of dollars per minute. 
AeroAgent uses a modular AI-SQL pipeline to automate part identification and quoting.

## 1. The Architecture
AeroAgent is built on three core pillars:
1. **Extraction:** Converting chaotic human emails into structured JSON.
2. **Routing:** Validating inventory, alternate parts, and aviation certificates (EASA/FAA).
3. **Generation:** Drafting a human-like, professional response.

In [ ]:
# Import our modular source code
import sys
sys.path.append('..')
from src import database, extractor, router, generator
from google.colab import userdata

# Initialize the database
database.initialize_database()
print('Database Initialized with Synthetic MRO Data')

## 2. Testing the Logic: Alternate Part Fallback
In this scenario, we request a part (`VLV-1010`) that is out of stock. 
The system should automatically find its approved alternate (`VLV-1010-MOD`).

In [ ]:
api_key = userdata.get('GEMINI_API_KEY')
client = extractor.get_ai_client(api_key)

sample_request = "Urgent request for VLV-1010 in OH condition."

# 1. Extract
extracted = extractor.extract_aog_data(sample_request, client)

# 2. Route (This is where the 'Alternate' logic triggers)
decision = router.route_aog_request('../data/processed/aeroagent.db', extracted)

print(f"Original Request: {extracted['part_number']}")
print(f"Decision Status: {decision['status']}")
print(f"Match Type: {decision.get('match_type')}")
print(f"Offered Part: {decision.get('part_number')}")

## 3. Final Output Generation
The final step uses the decision above to draft an email.

In [ ]:
email = generator.draft_quote_email(decision, client)
print("FINAL DRAFT ")
print(email)